In [28]:
import time
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pymatgen.core import Structure
from mofdscribe.featurizers.chemistry.racs import RACS

CIF_DIR = Path("/Users/hannah/Desktop/ethz/Semester6/dc-mof-project/data/cif_structures")
PARQUET = "/Users/hannah/Desktop/ethz/Semester6/dc-mof-project/data/cleaned/hmof_dataset_cleaned.parquet"
OUT_PATH = Path("/Users/hannah/Desktop/ethz/Semester6/dc-mof-project/data/rac_features.parquet")
CHECKPOINT_EVERY = 50

cleaned_names = set(pq.read_table(PARQUET, columns=["name"]).column("name").to_pylist())
all_cifs = sorted(p for p in CIF_DIR.glob("*.cif") if p.stem in cleaned_names)

# Resume: skip already-processed names
done_names = set()
if OUT_PATH.exists():
    done_names = set(pq.read_table(OUT_PATH, columns=["name"]).column("name").to_pylist())
    print(f"Resuming: {len(done_names)} already done")

cif_files = [p for p in all_cifs if p.stem not in done_names]
print(f"Remaining: {len(cif_files)} / {len(all_cifs)}")

racs = RACS()
feature_labels = racs.feature_labels()

def save_checkpoint(records):
    if not records:
        return
    new_df = pd.DataFrame(records)
    new_ok = new_df[new_df.status == "ok"].drop(columns=["time_s", "n_atoms", "status"])
    new_table = pa.Table.from_pandas(new_ok, preserve_index=False)
    if OUT_PATH.exists():
        existing = pq.read_table(OUT_PATH)
        combined = pa.concat_tables([existing, new_table])
        pq.write_table(combined, OUT_PATH)
    else:
        pq.write_table(new_table, OUT_PATH)

records = []
for i, cif_path in enumerate(cif_files):
    try:
        structure = Structure.from_file(str(cif_path))
        t0 = time.perf_counter()
        feats = racs.featurize(structure)
        elapsed = time.perf_counter() - t0
        row = {"name": cif_path.stem, "time_s": elapsed, "n_atoms": len(structure), "status": "ok"}
        row.update(dict(zip(feature_labels, feats)))
        records.append(row)
    except Exception as e:
        records.append({"name": cif_path.stem, "time_s": np.nan, "n_atoms": np.nan, "status": str(e)})

    if (i + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(records)
        records = []
        print(f"  checkpoint at {i + 1 + len(done_names)} / {len(all_cifs)}")

# Save any remaining
save_checkpoint(records)
total_done = pq.read_table(OUT_PATH).num_rows
print(f"Done. Total in file: {total_done}")



Remaining: 111469 / 111469


/Users/hannah/anaconda3/envs/mofdscribe/lib/python3.10/site-packages/structuregraph_helpers/create.py:101: FutureWarning: with_local_env_strategy is deprecated, and will be removed on 2025-03-20
Use from_local_env_strategy in pymatgen.analysis.graphs instead.
Deprecated on 2024-03-29.
  sg = StructureGraph.with_local_env_strategy(structure, get_local_env_method(method))


KeyboardInterrupt: 

In [29]:
print("Computed Features:", feature_labels)

Computed Features: ['racs_bb-linker_all_prop-I_scope-1_propagg-diff_corragg-avg_bbagg-avg', 'racs_bb-linker_all_prop-I_scope-1_propagg-diff_corragg-avg_bbagg-sum', 'racs_bb-linker_all_prop-I_scope-1_propagg-diff_corragg-sum_bbagg-avg', 'racs_bb-linker_all_prop-I_scope-1_propagg-diff_corragg-sum_bbagg-sum', 'racs_bb-linker_all_prop-I_scope-1_propagg-product_corragg-avg_bbagg-avg', 'racs_bb-linker_all_prop-I_scope-1_propagg-product_corragg-avg_bbagg-sum', 'racs_bb-linker_all_prop-I_scope-1_propagg-product_corragg-sum_bbagg-avg', 'racs_bb-linker_all_prop-I_scope-1_propagg-product_corragg-sum_bbagg-sum', 'racs_bb-linker_all_prop-I_scope-2_propagg-diff_corragg-avg_bbagg-avg', 'racs_bb-linker_all_prop-I_scope-2_propagg-diff_corragg-avg_bbagg-sum', 'racs_bb-linker_all_prop-I_scope-2_propagg-diff_corragg-sum_bbagg-avg', 'racs_bb-linker_all_prop-I_scope-2_propagg-diff_corragg-sum_bbagg-sum', 'racs_bb-linker_all_prop-I_scope-2_propagg-product_corragg-avg_bbagg-avg', 'racs_bb-linker_all_prop-I_sc

In [27]:
import pandas as pd

# Load the Parquet file
file_path = "/Users/hannah/Desktop/ethz/Semester6/dc-mof-project/rac_features.parquet"

# Read the file into a DataFrame
df = pd.read_parquet(file_path, engine="fastparquet")

# Display the structure (columns and data types)
print("Columns and Data Types:")
print(df.dtypes)

# Optionally, display the first few rows to understand the data
print("\nFirst 5 Rows:")
print(df.head())

Columns and Data Types:
name                                                                              object
racs_bb-linker_all_prop-I_scope-1_propagg-diff_corragg-avg_bbagg-avg             float64
racs_bb-linker_all_prop-I_scope-1_propagg-diff_corragg-avg_bbagg-sum             float64
racs_bb-linker_all_prop-I_scope-1_propagg-diff_corragg-sum_bbagg-avg             float64
racs_bb-linker_all_prop-I_scope-1_propagg-diff_corragg-sum_bbagg-sum             float64
                                                                                  ...   
racs_bb-nodes_prop-mod_pettifor_scope-3_propagg-diff_corragg-sum_bbagg-sum       float64
racs_bb-nodes_prop-mod_pettifor_scope-3_propagg-product_corragg-avg_bbagg-avg    float64
racs_bb-nodes_prop-mod_pettifor_scope-3_propagg-product_corragg-avg_bbagg-sum    float64
racs_bb-nodes_prop-mod_pettifor_scope-3_propagg-product_corragg-sum_bbagg-avg    float64
racs_bb-nodes_prop-mod_pettifor_scope-3_propagg-product_corragg-sum_bbagg-sum    float